In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier
import joblib  # For saving models
import time  # For measuring computational time

# Load the dataset
data = pd.read_csv("Synthetic Frailty Data.csv")  # Replace with your dataset's file path

# Preprocess the data
label_encoder = LabelEncoder()

# Map 'Standing on One Leg Result' to numeric values
data['Standing on One Leg Result'] = label_encoder.fit_transform(data['Standing on One Leg Result'])

# Define input features (X) and output target (y)
X = data[['Age', 'Standing on One Leg with Eye Open (s)']]
y = data['Standing on One Leg Result']

# Standardize features for models like SVM and KNN
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

joblib.dump(scaler, 'Scaler_SOOLWEO.pkl')
print("Scaler saved as 'Scaler.pkl'")
joblib.dump(label_encoder, 'Label_Encoder_SOOLWEO.pkl')  # Save the LabelEncoder
print("Label Encoder saved as 'Label_Encoder_SOOLWEO.pkl'")

# Initialize models
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Support Vector Machine": SVC(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "XGBoost": XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
}

# Train, evaluate, and save models
results = {}
computation_costs = {}

for model_name, model in models.items():
    start_time = time.time()  # Start timing
    model.fit(X_train, y_train)
    end_time = time.time()  # End timing

    # Predictions
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    # Computational cost (training time in seconds)
    training_time = end_time - start_time
    computation_costs[model_name] = training_time

    # Print model results
    print(f"Model: {model_name}")
    print(f"Accuracy: {acc:.2f}")
    print(f"Training Time (seconds): {training_time:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
    print("-" * 50)
    results[model_name] = acc

    # Save the model as a .pkl file
    file_name = f"{model_name.replace(' ', '_')}_SOOLWEO.pkl"  # Append SOOLWEO to the file name
    joblib.dump(model, file_name)
    print(f"Model '{model_name}' saved as '{file_name}'")

# Compare accuracies and computational costs
print("\nModel Performance Summary:")
for model_name, acc in results.items():
    print(f"{model_name}: Accuracy = {acc:.2f}, Training Time = {computation_costs[model_name]:.4f} seconds")


Scaler saved as 'Scaler.pkl'
Label Encoder saved as 'Label_Encoder_SOOLWEO.pkl'
Model: Logistic Regression
Accuracy: 0.98
Training Time (seconds): 0.0650

Classification Report:
                               precision    recall  f1-score   support

                         Fail       0.98      0.96      0.97        54
                         Pass       1.00      1.00      1.00        98
                 Risk of Fall       1.00      1.00      1.00        33
Risk of Locomotor Instability       0.88      0.93      0.90        15

                     accuracy                           0.98       200
                    macro avg       0.96      0.97      0.97       200
                 weighted avg       0.99      0.98      0.99       200

--------------------------------------------------
Model 'Logistic Regression' saved as 'Logistic_Regression_SOOLWEO.pkl'
Model: Random Forest
Accuracy: 1.00
Training Time (seconds): 0.6233

Classification Report:
                               precis

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:36:51] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Model: XGBoost
Accuracy: 0.99
Training Time (seconds): 2.4611

Classification Report:
                               precision    recall  f1-score   support

                         Fail       1.00      0.98      0.99        54
                         Pass       1.00      1.00      1.00        98
                 Risk of Fall       1.00      1.00      1.00        33
Risk of Locomotor Instability       0.94      1.00      0.97        15

                     accuracy                           0.99       200
                    macro avg       0.98      1.00      0.99       200
                 weighted avg       1.00      0.99      1.00       200

--------------------------------------------------
Model 'XGBoost' saved as 'XGBoost_SOOLWEO.pkl'

Model Performance Summary:
Logistic Regression: Accuracy = 0.98, Training Time = 0.0650 seconds
Random Forest: Accuracy = 1.00, Training Time = 0.6233 seconds
Support Vector Machine: Accuracy = 0.99, Training Time = 0.0223 seconds
K-Nearest Nei

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile

# Create a zip archive containing the models, scaler, and label encoder
files_to_download = [
    "Logistic_Regression_SOOLWEO.pkl", "Random_Forest_SOOLWEO.pkl", "Support_Vector_Machine_SOOLWEO.pkl",
    "K-Nearest_Neighbors_SOOLWEO.pkl", "XGBoost_SOOLWEO.pkl", "Scaler_SOOLWEO.pkl", "Label_Encoder_SOOLWEO.pkl"
]

with zipfile.ZipFile('SOOLWEO_Models.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in files_to_download:
        zipf.write(file)

print("Download the 'SOOLWEO_Models.zip' file from your directory.")


Download the 'SOOLWEO_Models.zip' file from your directory.


In [ ]:
#Apna kaam beshak ignore it please

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.preprocessing import StandardScaler

# Load the dataset
file_path = "Physical_Frailty_Dataset_Kyrgyzstan.xlsx"
data = pd.read_excel(file_path)

# Extract relevant columns
test_data = data[["Age", "Standing on One Leg with Eye Open (s)"]]

# Standardize the test data using the pre-fitted scaler
test_data_scaled = scaler.transform(test_data)

# Initialize a dictionary to store computational costs
computational_costs = {}

# Initialize a DataFrame to store predictions
predictions_df = test_data.copy()

# Make predictions with each model and measure computational cost
for model_name, model in models.items():
    start_time = time.time()  # Start timer for the entire dataset
    batch_predictions = model.predict(test_data_scaled)  # Predict for the entire dataset
    end_time = time.time()  # End timer for the entire dataset

    # Save predictions in the DataFrame
    predictions_df[model_name] = label_encoder.inverse_transform(batch_predictions)

    # Measure computational cost per example
    avg_time_per_example = (end_time - start_time) / len(test_data)
    total_time = end_time - start_time

    # Store computational costs
    computational_costs[model_name] = {
        "Time per example (average)": avg_time_per_example,
        "Total time (whole dataset)": total_time,
    }

# Save the predictions DataFrame to an Excel file
output_file_path = "Physical_Frailty_Dataset_Predictions.xlsx"
predictions_df.to_excel(output_file_path, index=False)

# Print computational costs
print("\nComputational Costs:")
for model_name, costs in computational_costs.items():
    print(f"\nModel: {model_name}")
    print(f"  Time per example (average): {costs['Time per example (average)']:.6f} seconds")
    print(f"  Total time (whole dataset): {costs['Total time (whole dataset)']:.6f} seconds")

# Provide download link for the updated Excel file
print(f"\nPredictions saved to: {output_file_path}")



Computational Costs:

Model: Logistic Regression
  Time per example (average): 0.000001 seconds
  Total time (whole dataset): 0.000332 seconds

Model: Random Forest
  Time per example (average): 0.000030 seconds
  Total time (whole dataset): 0.007580 seconds

Model: Support Vector Machine
  Time per example (average): 0.000011 seconds
  Total time (whole dataset): 0.002733 seconds

Model: K-Nearest Neighbors
  Time per example (average): 0.000051 seconds
  Total time (whole dataset): 0.012889 seconds

Model: XGBoost
  Time per example (average): 0.000005 seconds
  Total time (whole dataset): 0.001364 seconds

Predictions saved to: Physical_Frailty_Dataset_Predictions.xlsx


In [ ]:
# Load necessary libraries
import pandas as pd
import joblib

# Load the saved scaler and label encoder
scaler = joblib.load('Scaler_SOOLWEO.pkl')
label_encoder = joblib.load('Label_Encoder_SOOLWEO.pkl')

# Example dataset
example_data = pd.DataFrame({
    'Age': [65, 80, 70],  # Example ages
    'Standing on One Leg with Eye Open (s)': [20, 9.9, 80]  # Example test durations
})

# Preprocess the example data
example_data_scaled = scaler.transform(example_data)

# Load trained models
models = {
    "Logistic Regression": joblib.load("Logistic_Regression_SOOLWEO.pkl"),
    "Random Forest": joblib.load("Random_Forest_SOOLWEO.pkl"),
    "Support Vector Machine": joblib.load("Support_Vector_Machine_SOOLWEO.pkl"),
    "K-Nearest Neighbors": joblib.load("K-Nearest_Neighbors_SOOLWEO.pkl"),
    "XGBoost": joblib.load("XGBoost_SOOLWEO.pkl"),
}

# Predict results with each model
predictions = {}
for model_name, model in models.items():
    pred_numeric = model.predict(example_data_scaled)  # Numeric predictions
    pred_labels = label_encoder.inverse_transform(pred_numeric)  # Convert to original labels
    predictions[model_name] = pred_labels

# Display predictions
for model_name, pred_labels in predictions.items():
    print(f"Predictions using {model_name}:")
    for idx, label in enumerate(pred_labels):
        print(f"Example {idx + 1}: {label}")
    print("-" * 50)


Predictions using Logistic Regression:
Example 1: Pass
Example 2: Risk of Fall
Example 3: Pass
--------------------------------------------------
Predictions using Random Forest:
Example 1: Pass
Example 2: Risk of Fall
Example 3: Pass
--------------------------------------------------
Predictions using Support Vector Machine:
Example 1: Pass
Example 2: Risk of Fall
Example 3: Pass
--------------------------------------------------
Predictions using K-Nearest Neighbors:
Example 1: Pass
Example 2: Pass
Example 3: Pass
--------------------------------------------------
Predictions using XGBoost:
Example 1: Pass
Example 2: Pass
Example 3: Pass
--------------------------------------------------
